In [ ]:
from dnachisel import *
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import Bio.SeqIO as SeqIO
# import python_codon_tables as pct
#import glob
import json
from dnachisel.biotools import reverse_translate
import pandas as pd

In [ ]:
# now import fasta files
seq_dict = dict()
for i in SeqIO.parse('./example.faa',"fasta"):
    seq_dict[i.name] = i

In [ ]:
for i in seq_dict:
    print(i,seq_dict[i].seq,len(seq_dict[i]))

In [ ]:
dna_seqs = dict()
for i in seq_dict:
    dna_seqs[i] = reverse_translate(seq_dict[i]) + "TAA"

In [ ]:
for i in dna_seqs:
    # print(i,dna_seqs[i],len(dna_seqs[i])/(len(seq_dict[i])+1))
    print(len(dna_seqs[i])/(len(seq_dict[i])+1))


In [ ]:
output_seqs = dict()
constraint_summaries = list()
opt_summaries = list()
for i in dna_seqs:
    theprob = DnaOptimizationProblem(
        sequence = dna_seqs[i],
        constraints = [
            AvoidPattern("BsaI_site"),
            AvoidPattern("PmeI_site"),
            AvoidPattern("SapI_site"),
            AvoidPattern("BbsI_site"),
            AvoidPattern("PaqCI_site"),
            EnforceGCContent(mini=0.4,maxi=0.65,window=50),
            EnforceGCContent(mini=0.45,maxi=0.6),
            EnforceTranslation(start_codon="keep",translation = seq_dict[i] + "*"),
            AvoidRareCodons(min_frequency=0.1,species = 196627),
            AvoidHairpins(stem_size = 16, hairpin_window = 200)
        ],
        objectives=[
            CodonOptimize(method='match_codon_usage',species = 196627),
            UniquifyAllKmers(k=8)
        ]
    )
    theprob.max_random_iters=50000
    theprob.resolve_constraints();
    theprob.optimize();
    output_seqs[i] = SeqRecord(Seq(theprob.sequence),id = i)
    constraint_summaries.append(theprob.constraints_text_summary())
    opt_summaries.append(theprob.objectives_text_summary())
 #   clear_output(wait=True)
    

In [ ]:
for i in constraint_summaries:
    print(i)

In [ ]:
for i in opt_summaries:
    print(i)

In [ ]:
# for i in output_seqs:
#     SeqIO.write(SeqRecord(Seq(output_seqs[i]),i),"optimized_ORFs","fasta")

In [ ]:
forend = "GGTCTCAA"
revend = "CTGAGACC"

output_list_noends = list()
output_list_withends = list()
for i in output_seqs:
    output_list_noends.append(output_seqs[i])
    output_list_withends.append(forend + output_seqs[i] + revend)
SeqIO.write(output_list_noends,"ex_out.fna","fasta")
# SeqIO.write(output_list_withends,"optimized_seqs_withends.fa","fasta")